<a href="https://colab.research.google.com/github/shaziyarehamat07-cmd/intro-build-calculator/blob/main/intelligent_calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
%pip install -q 'langchain>=1,<2' 'langchain-groq<2'

In [26]:
import os
from getpass import getpass
from langchain_groq import ChatGroq

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ").strip()
if not os.environ["GROQ_API_KEY"]:
    raise ValueError("Enter a Groq API key in the hidden prompt.")
model = ChatGroq(model="openai/gpt-oss-20b", temperature=0, timeout=30, max_retries=1)
print("Model client configured. Cell 3 tests the connection.")

Model client configured. Cell 3 tests the connection.


In [27]:
APPLES = "One person brings five apples and another brings ten. They put them together. How many apples are there?"
plain_reply = model.invoke([("human", APPLES)])
print(plain_reply.content)

There are 15 apples in total.


In [28]:
role_reply = model.invoke([
    ("system", "You are a patient math tutor. Give one short sentence with the answer and its units."),
    ("human", APPLES),
])
print(role_reply.content)

There are 15 apples.


In [29]:
from langchain.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two whole-number quantities when they are combined."""
    return a + b

print(add.name)

add


In [30]:
from math import isfinite, radians, tan

@tool
def tree_distance(height_m: float, angle_deg: float) -> float | str:
    """Find horizontal tree distance on level ground from height above eye level in metres and angle of elevation in degrees."""
    if not isfinite(height_m) or height_m <= 0:
        return "Invalid height. Supply a positive finite height above eye level."
    if not isfinite(angle_deg) or not 0 < angle_deg < 90:
        return "Invalid angle. Supply an angle strictly between 0 and 90 degrees."
    return round(height_m / tan(radians(angle_deg)), 2)

print(tree_distance.name)

tree_distance


In [31]:
print("Apples:", add.invoke({"a": 5, "b": 10}))
print("Tree distance in metres:", tree_distance.invoke({"height_m": 10, "angle_deg": 30}))
print("Invalid angle:", tree_distance.invoke({"height_m": 10, "angle_deg": 0}))

Apples: 15
Tree distance in metres: 17.32
Invalid angle: Invalid angle. Supply an angle strictly between 0 and 90 degrees.


In [32]:
from langchain.agents import create_agent

INSTRUCTIONS = (
    "You are the Intelligent Calculator. Use add for combining two whole-number "
    "quantities. Use tree_distance only for horizontal distance on level ground "
    "when height above eye level in metres and angle of elevation in degrees "
    "are supplied. For supported complete problems, call the matching tool once "
    "and use its result. If a required value is missing, ask for it before "
    "calling a tool. Never guess inputs. If a tool reports an invalid input, ask "
    "for a correction. For other requests, explain what you can calculate. "
    "Give the final answer with units."
)
agent = create_agent(model=model, tools=[add, tree_distance], system_prompt=INSTRUCTIONS)
print("Agent ready")

Agent ready


In [33]:
from langchain_core.messages import ToolMessage

def solve(problem, *, run_name="intro_calculator"):
    return agent.invoke(
        {"messages": [("human", problem)]},
        config={"recursion_limit": 8, "run_name": run_name},
    )

def show_run(problem):
    result = solve(problem)
    call_count = 0
    for message in result["messages"]:
        for call in getattr(message, "tool_calls", []):
            call_count += 1
            print("Tool requested:", call["name"], call["args"])
        if isinstance(message, ToolMessage):
            print("Tool returned:", message.name, message.content)
    print("Tool requests:", call_count)
    print("Final answer:", result["messages"][-1].content)
    return result

print("Viewer ready")

Viewer ready


In [34]:
apples_run = show_run(APPLES)

Tool requested: add {'a': 5, 'b': 10}
Tool returned: add 15
Tool requests: 1
Final answer: There are **15 apples**.


In [35]:
TREE = (
    "On level ground, the top of a tree is 10 metres above my eye level. "
    "The angle of elevation to the top is 30 degrees. "
    "How far away is the tree horizontally?"
)
tree_run = show_run(TREE)

Tool requested: tree_distance {'angle_deg': 30, 'height_m': 10}
Tool returned: tree_distance 17.32
Tool requests: 1
Final answer: The horizontal distance from you to the tree is **17.32 m**.


In [36]:
MISSING_ANGLE = (
    "On level ground, the top of a tree is 10 metres above my eye level. "
    "How far away is the tree horizontally?"
)
missing_run = show_run(MISSING_ANGLE)

Tool requests: 0
Final answer: I can calculate the horizontal distance using the tree_distance tool, but I need the angle of elevation (in degrees). Could you please provide that?


In [37]:
from langsmith import Client, tracing_context

destination = input("LangSmith destination type (agent/project): ").strip().lower()
if destination == "agent":
    os.environ.pop("LANGSMITH_PROJECT", None)
    os.environ["LANGSMITH_AGENT_ID"] = input("Agent ID: ").strip()
    os.environ["LANGSMITH_AGENT_ENVIRONMENT"] = input(
        "Environment (for example local): "
    ).strip()
elif destination == "project":
    os.environ.pop("LANGSMITH_AGENT_ID", None)
    os.environ.pop("LANGSMITH_AGENT_ENVIRONMENT", None)
    os.environ["LANGSMITH_PROJECT"] = input("Project name: ").strip()
else:
    raise ValueError("Enter agent or project, then rerun this cell.")

endpoint = input("LangSmith API URL (blank for default US): ").strip()
if endpoint:
    os.environ["LANGSMITH_ENDPOINT"] = endpoint
workspace_id = input("Workspace ID (blank if unnecessary): ").strip()
if workspace_id:
    os.environ["LANGSMITH_WORKSPACE_ID"] = workspace_id
smith_key = getpass("LangSmith API key: ").strip()
smith_client = Client(api_key=smith_key)
print("Tracing client configured.")

LangSmith destination type (agent/project): agent
Agent ID: fleet-1157c55b1f36453090ccc83212134c63
Environment (for example local): local
LangSmith API URL (blank for default US): 
Workspace ID (blank if unnecessary): 
LangSmith API key: ··········
Tracing client configured.


In [38]:
with tracing_context(enabled=True, client=smith_client):
    solve(APPLES, run_name="intro_apples")
    solve(TREE, run_name="intro_tree")
    solve(MISSING_ANGLE, run_name="intro_missing_angle")
smith_client.flush()
print("Look for intro_apples, intro_tree and intro_missing_angle.")

Look for intro_apples, intro_tree and intro_missing_angle.


In [39]:
fresh_run = show_run('One person brings seven apples and another brings eight. How many apples do they have together?')

Tool requested: add {'a': 7, 'b': 8}
Tool returned: add 15
Tool requests: 1
Final answer: They have a total of **15 apples**.


In [40]:
projects = list(smith_client.list_projects())

print("Available LangSmith projects:")

for project in projects:
    print("-", project.name)

Available LangSmith projects:
- fleet-1157c55b1f36453090ccc83212134c63-local
- fleet-1157c55b1f36453090ccc83212134c63-production
- gateway


In [41]:
for project in projects:
    print("\nPROJECT:", project.name)

    runs = list(
        smith_client.list_runs(
            project_name=project.name,
            limit=50
        )
    )

    print("Runs found:", len(runs))

    for run in runs:
        print(
            "Run:", run.name,
            "| Type:", run.run_type,
            "| Status:", run.status
        )


PROJECT: fleet-1157c55b1f36453090ccc83212134c63-local


/tmp/ipykernel_34510/2496013403.py:5: LangSmithDeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  smith_client.list_runs(


Runs found: 34
Run: ChatGroq | Type: llm | Status: success
Run: model | Type: chain | Status: success
Run: intro_missing_angle | Type: chain | Status: success
Run: ChatGroq | Type: llm | Status: success
Run: model | Type: chain | Status: success
Run: tree_distance | Type: tool | Status: success
Run: tools | Type: chain | Status: success
Run: ChatGroq | Type: llm | Status: success
Run: model | Type: chain | Status: success
Run: intro_tree | Type: chain | Status: success
Run: ChatGroq | Type: llm | Status: success
Run: model | Type: chain | Status: success
Run: add | Type: tool | Status: success
Run: tools | Type: chain | Status: success
Run: ChatGroq | Type: llm | Status: success
Run: model | Type: chain | Status: success
Run: intro_apples | Type: chain | Status: success
Run: ChatGroq | Type: llm | Status: success
Run: model | Type: chain | Status: success
Run: intro_missing_angle | Type: chain | Status: success
Run: ChatGroq | Type: llm | Status: success
Run: model | Type: chain | Stat

In [42]:
runs = list(
    smith_client.list_runs(
        project_name=projects[0].name,
        limit=100
    )
)

target_runs = [
    "intro_apples",
    "intro_tree",
    "intro_missing_angle"
]

for name in target_runs:
    matches = [run for run in runs if run.name == name]

    if matches:
        for run in matches:
            print(
                "Run:", run.name,
                "| Type:", run.run_type,
                "| Status:", run.status,
                "| Error:", run.error
            )
    else:
        print("NOT FOUND:", name)

/tmp/ipykernel_34510/3215990977.py:2: LangSmithDeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead.See https://docs.langchain.com/langsmith/smithdb-sdk-migration-query-runs#runs-query for the migration guide.
  smith_client.list_runs(


Run: intro_apples | Type: chain | Status: success | Error: None
Run: intro_apples | Type: chain | Status: success | Error: None
Run: intro_tree | Type: chain | Status: success | Error: None
Run: intro_tree | Type: chain | Status: success | Error: None
Run: intro_missing_angle | Type: chain | Status: success | Error: None
Run: intro_missing_angle | Type: chain | Status: success | Error: None


# Intelligent Calculator – Build Explanation

## 1. Project Overview

This project demonstrates an Intelligent Calculator using Python, LangChain, Groq and the Model Context Protocol (MCP).

The agent understands mathematical word problems and selects the appropriate tool to solve them.

## 2. Tool Choices

### Addition Tool – `add`
I created an addition tool to combine two whole-number quantities. The agent extracts the two numbers from the problem and calls this tool to calculate their sum.

### Tree Distance Tool – `tree_distance`
I created a tool using trigonometry to calculate horizontal distance on level ground. It uses the height above eye level and angle of elevation.

The formula is:

Distance = Height / tan(Angle)

## 3. Test Observations

- Apples test (5 + 10): The agent called `add` with arguments 5 and 10. The tool returned 15.
- Tree test: The agent called `tree_distance` with height 10 metres and angle 30 degrees. The tool returned 17.32 metres.
- Missing-angle test: The agent did not call the tree-distance tool. It asked the user to provide the missing angle instead of guessing.
- Fresh test (7 + 8): The agent called `add` with arguments 7 and 8. The tool returned 15.

## 4. LangSmith Observations

**Status: Configured and test runs submitted; dashboard verification not documented here.**

LangSmith tracing was configured, and the apples, tree and missing-angle runs were submitted using the tracing context. The notebook does not include a screenshot or exported trace proving their appearance in the LangSmith dashboard.

## 5. MCP Inspector Observations

**Status: Demonstrated.**

The Intelligent Calculator MCP server was started locally and connected through MCP Inspector using Streamable HTTP.

The `solve_math_problem` tool was discovered and tested successfully. The addition test returned 15 apples.

## 6. OpenAI Secure MCP Tunnel

**Status: Demonstrated locally.**

The tunnel profile was configured and the diagnostic check returned `RESULT ok`. The local dashboard showed Health: Live, Ready: Ready and Logs: Connected.

OAuth discovery displayed HTTP 404 errors with the no-auth sample configuration.

## 7. ChatGPT Integration

**Status: Not completed.**

The custom MCP connection option was unavailable in my ChatGPT account. Therefore, I could not demonstrate a tool call directly from ChatGPT. This is recorded as an access limitation, not a successful ChatGPT integration.

## 8. Conclusion

This project demonstrates how an AI agent can understand mathematical problems, select tools, execute calculations and request missing information instead of making assumptions. It also demonstrates local MCP server testing, tracing configuration and secure tunnel setup.



## 9. MCP Evidence Availability (Reviewer Note)

The notebook contains saved execution outputs for the four calculator demonstrations and saved LangSmith run-query output. The MCP Inspector and tunnel checks were performed locally during setup, but their terminal/browser screenshots or raw logs were not captured as notebook cell outputs. I am therefore not presenting them as notebook-verified execution evidence.

- **MCP Inspector:** Locally demonstrated: the Streamable HTTP MCP server was reachable, `solve_math_problem` was discovered, and the addition test returned 15. The raw Inspector result is not embedded in this notebook.
- **OpenAI Secure MCP Tunnel:** Locally demonstrated: the diagnostic reported `RESULT OK`; the local dashboard showed Health `Live` and Ready `Ready`. The raw diagnostic/dashboard output is not embedded in this notebook. OAuth discovery returned HTTP 404 with the no-auth sample profile; this was recorded as a configuration limitation, not a successful OAuth check.
- **ChatGPT custom MCP call:** Not completed; the required custom-connector option was unavailable in the account used for this build.

These distinctions are intentional: locally performed checks are described as such, and checks without saved notebook evidence are not claimed as reproducible notebook outputs.


In [ ]:
# FINAL LIVE EVIDENCE CHECK
# Run this cell after the setup/configuration cells above.
# It re-runs all four demonstrations and prints the LangSmith run statuses.

from langchain import __version__ as langchain_version

print("=== APPLES: 5 + 10 ===")
with tracing_context(enabled=True, client=smith_client):
    apples_evidence = show_run(APPLES)

print("\n=== TREE DISTANCE: 10 m at 30 degrees ===")
with tracing_context(enabled=True, client=smith_client):
    tree_evidence = show_run(TREE)

print("\n=== MISSING ANGLE ===")
with tracing_context(enabled=True, client=smith_client):
    missing_angle_evidence = show_run(MISSING_ANGLE)

print("\n=== FRESH APPLES: 7 + 8 ===")
fresh_evidence = show_run(
    "One person brings seven apples and another brings eight. How many apples do they have together?"
)

smith_client.flush()
print("\n=== LANGSMITH RUN VERIFICATION ===")
verification_runs = list(
    smith_client.list_runs(project_name=projects[0].name, limit=100)
)
for run_name in ["intro_apples", "intro_tree", "intro_missing_angle"]:
    matches = [run for run in verification_runs if run.name == run_name]
    if matches:
        for run in matches:
            print(f"{run.name} | {run.run_type} | {run.status} | error={run.error}")
    else:
        print(f"{run_name} | NOT FOUND")
